In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import scipy.stats as stats
import warnings
import plotly.express as px
from scipy.stats import pearsonr
import io
from scipy.stats import pearsonr, ttest_ind, f_oneway
from sklearn.linear_model import LinearRegression
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go


df = pd.read_csv("data/cleaned/cleaned_sale_properties.csv")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
df["has_parking"] = df["has_parking"].map({True: 1, False: 0})
df["garden"] = df["garden"].map({True: 1, False: 0})
df["terrace"] = df["terrace"].map({True: 1, False: 0})



df["epc"] = df["epc"].replace({
    "A++": "A"
})


epc_mapping = {
    "A": 7,
    "B": 6,
    "C": 5,
    "D": 4,
    "E": 3,
    "F": 2,
    "G": 1
}

df["epc_numeric"] = df["epc"].map(epc_mapping)

epc_colors = [
    "#004E00", "#006400", "#228B22",
    "#9ACD32", "#FFF700", "#FFA500",
    "#FF4500", "#D2691E", "#8B4513"
]
df["nearest_city"]
top_cities = df["nearest_city"].value_counts().head(5).index
df_top = df[df["nearest_city"].isin(top_cities)]

df_apartment = df[df["category"].str.lower() == "apartment"]
df_house = df[df["category"].str.lower() == "house"]

In [ ]:
def compute_correlations(df, target="price"):

    results = []

    for col in df.columns:
        if col == target:
            continue

        try:
            temp = df[[col, target]].dropna()

            if len(temp) < 10:
                continue

            x = pd.to_numeric(temp[col], errors="coerce")
            y = temp[target]

            valid = x.notna() & y.notna()
            x = x[valid]
            y = y[valid]

            if x.nunique() < 2:
                continue

            r, p = pearsonr(x, y)

            results.append({
                "feature": col,
                "r": r,
                "p": p
            })

        except:
            continue

    return pd.DataFrame(results)


def filter_features(results_df, r_threshold=0.10, p_threshold=0.05):

    filtered = results_df[
        (results_df["r"].abs() >= r_threshold) &
        (results_df["p"] < p_threshold)
    ]

    return filtered["feature"].tolist()


def build_filtered_df(df, features):

    cols = features + ["price"]
    df_filtered = df[cols].copy()

    return df_filtered


def plot_filtered_heatmaps(df_house, df_apartment):

    # correlations
    corr1 = df_house.corr(numeric_only=True)
    corr1 = corr1.dropna(axis=0, how="all").dropna(axis=1, how="all")

    corr2 = df_apartment.corr(numeric_only=True)
    corr2 = corr2.dropna(axis=0, how="all").dropna(axis=1, how="all")

    # subplot
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Houses (filtered)", "Apartments (filtered)"),
        horizontal_spacing=0.15              
    )

    # heatmap 1
    fig.add_trace(
        go.Heatmap(
            z=corr1.values,
            x=corr1.columns,
            y=corr1.index,
            colorscale="RdBu_r",
            zmin=-1,
            zmax=1
        ),
        row=1, col=1
    )

    # heatmap 2
    fig.add_trace(
        go.Heatmap(
            z=corr2.values,
            x=corr2.columns,
            y=corr2.index,
            colorscale="RdBu_r",
            zmin=-1,
            zmax=1
        ),
        row=1, col=2
    )

    fig.update_layout(
        height=600,
        width=1200,
        title="Correlation Heatmaps (Only Significant Features)"
    )

    fig.show()

df_house = df[df["category"].str.lower() == "house"]
df_apartment = df[df["category"].str.lower() == "apartment"]

# compute correlations
results_house = compute_correlations(df_house)
results_apartment = compute_correlations(df_apartment)

# filter features
features_house = filter_features(results_house)
features_apartment = filter_features(results_apartment)

print("House features:", features_house)
print("Apartment features:", features_apartment)

# build filtered datasets
df_house_filtered = build_filtered_df(df_house, features_house)
df_apartment_filtered = build_filtered_df(df_apartment, features_apartment)

# plot
plot_filtered_heatmaps(df_house_filtered, df_apartment_filtered)

In [ ]:
def filter_features(results_df, r_threshold=0.10, p_threshold=0.05):

    filtered = results_df[
        (results_df["r"].abs() >= r_threshold) &
        (results_df["p"] < p_threshold)
    ]

    return filtered["feature"].tolist()


features_house = filter_features(results_house)
features_apartment = filter_features(results_apartment)

In [ ]:
def plot_interactive_heatmap(df, title):

    corr = df.corr(numeric_only=True)

    corr = corr.dropna(axis=0, how="all").dropna(axis=1, how="all")

    fig = px.imshow(
        corr,
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        title=title
    )

    fig.update_layout(height=700, width=800)

    fig.show()

In [ ]:
def scatter_surface_price(df, title):
    plt.figure(figsize = (10,6))

    sns.scatterplot(
        data = df,
        x = "livable_surface",
        y = "price",
        hue = "category",
        alpha = 0.5 
    )
    plt.yscale("log")
    ax = plt.gca()
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'€{x/1e6:.1f}M')
    )

    plt.title(title)
    plt.xlabel("Livable Surface (m²)")
    plt.ylabel("Price (€ millions)")
    plt.show()

scatter_surface_price(df_apartment, "Scatter_apartment")

In [ ]:
def get_good_features(results_df):
    return results_df[
        (results_df["correlation"].abs() > 0.2) &
        (results_df["p_value"] < 0.05)
    ]["feature"].tolist()


def get_bad_features(results_df):
    return results_df[
        (results_df["correlation"].abs() < 0.1) &
        (results_df["p_value"] > 0.05)
    ]["feature"].tolist()
good_features = get_good_features(results_df)
bad_features = get_bad_features(results_df)

print("\n Good features:\n", good_features)
print("\n Bad features:\n", bad_features)

results_df.to_csv("feature_ranking.csv", index=False)


pd.DataFrame(good_features, columns=["feature"]).to_csv(
    "good_features.csv", index=False
)


pd.DataFrame(bad_features, columns=["feature"]).to_csv(
    "bad_features.csv", index=False
)


with open("eda_results_after.txt", "w") as f:
    
    for col in df.columns:
        if col != "price":
            analyze_feature(df, col, f)

print("Results saved to eda_results.txt")

In [ ]:
def regression_pipline(df, target = "price", r_threshold = 0.15, p_threshold = 0.05):
    results = []
    
    for col in df.columns:
        if col == target:
            continue

        try:
            temp = df[[col, target]].dropna()

            if len(temp) < 10:
                continue 

            x = pd.to_numeric(temp[col], errors= "coerce")
            y = temp[target]

            valid = x.notna() & y.notna()
            x = x[valid]
            y = y[valid]

            if len(x) < 2:
                continue
            corr, p_value = pearsonr(x, y)

            results.append({
                "feature":col,
                "correlation":corr,
                "p_value": p_value
            })
    
        except:
            continue
    results_df = pd.DataFrame(results)

    print("\nTop correlations:")
    print(results_df.sort_values(by="correlation", 
                                key=abs, ascending=False).head(10))
    selected = results_df[
        (results_df["correlation"].abs() >=r_threshold) &
        (results_df["p_value"] < p_threshold)
    ]
    features = selected["feature"].tolist()
    
    bad_features = ["postal_code"]
    features = [f for f in features if f not in bad_features]

    print("\nSelected features:", features)

    if len(features) == 0:
        print("No features selected — try lowering threshold")
        return None
    
    df = df[df[target] > 0]
    x = df[features]
    y = np.log(df[target])

    df_model = x.join(y.rename("price_log"))
    df_model = df_model.dropna(subset=["price_log"])

    x = df_model[features].fillna(0)
    y = df_model["price_log"]

    print("\nFinal dataset shape:", x.shape)

    model = LinearRegression()
    model.fit(x,y)
    r2 = model.score(x, y)

    print("\nR² score:", r2)
    
    n = x.shape[0]
    p = x.shape[1]
    
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    print("Adjusted R²:", adj_r2)


    print("\nCoefficients:")
    for f, coef in zip(features, model.coef_):
        print(f"{f}: {coef:.4f}")

    return model, x, y

In [ ]:
def plot_heatmap(df, title):

    corr = df.corr(numeric_only = True)
    corr = corr.dropna(axis=0, how="all").dropna(axis=1, how="all")
    plt.figure(figsize=(10,8))
    sns.heatmap(corr, 
                cmap = "coolwarm", 
                center = 0,
                annot = True,
                fmt = ".2f",
                annot_kws={"size":8})

    plt.title(title)
    plt.show()


    


In [ ]:
def price_map():
        """Geographic scatter of price per m2 across Belgium."""
        geo = df.dropna(subset=["latitude", "longitude", "price_per_sqm"])
        lo, hi = geo["price_per_sqm"].quantile([0.01, 0.99])   
        geo = geo[(geo["price_per_sqm"] >= lo) & (geo["price_per_sqm"] <= hi)]

        
        plt.figure(figsize=(11, 11))
        sc = plt.scatter(geo["longitude"], geo["latitude"], c=geo["price_per_sqm"],
                         cmap="viridis", s=6, alpha=0.6)
        plt.colorbar(sc, label="Price per m2 (EUR)")
        plt.title("Price per m2 map (Belgium)", fontsize=15, fontweight="bold")
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.gca().set_aspect("equal", adjustable="datalim")
        plt.show()

In [ ]:
def plot_interactive_heatmap(df, title):

    # compute correlation
    corr = df.corr(numeric_only=True)

    # remove empty rows/cols
    corr = corr.dropna(axis=0, how="all").dropna(axis=1, how="all")

    # create heatmap
    fig = px.imshow(
        corr,
        color_continuous_scale="RdBu_r",   
        zmin=-1,
        zmax=1,
        title=title
    )

    fig.update_layout(height=800, width=900)

    fig.show()


df["nearest_city"]

In [ ]:
def box_plot_cities_with_values_sqm(df, title):
    plt.figure(figsize=(12,6))
    top_cities = df["nearest_city"].value_counts().head(5).index
    
    

    palette = {
         "apartment": epc_colors[2],
         "house": epc_colors[6]}
    df_top = df[df["nearest_city"].isin(top_cities)]
    ax = sns.boxplot(
        data =df_top,
        x="nearest_city",
        y = "price_per_sqm",
        hue= "category",
        palette = palette,
        showfliers=False 
    )
    cities_order = [t.get_text() for t in ax.get_xticklabels()]
    grouped = df_top.groupby(["nearest_city", "category"])["price_per_sqm"]
    
    for i, ((city, cat), values) in enumerate(grouped):
        q1 = values.quantile(0.25)
        median = values.quantile(0.5)
        q3 = values.quantile(0.75)

        x_pos = cities_order.index(city)

        if cat == "apartment":
            offset = -0.2
        else:
            offset = 0.2

        
        #ax.text(x_pos + offset, q1, f"{q1/1e6:.2f}M", ha="center", fontsize=7)
        ax.text(x_pos + offset, median, f"{median:.0f}EUR/m", ha="center", fontsize=8, color="white")
        #ax.text(x_pos + offset, q3, f"{q3/1e6:.2f}M", ha="center", fontsize=7)

                      
        ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{int(x)} EUR')
    )
        """"
        #ax2 = ax.twinx()
        #counts = df_top["nearest_city"].value_counts().reindex(cities_order)
        #ax2.plot(
            counts.index,
            counts.values,
            color = "black",
            marker = "o",
            linestyle = "--"
        #)
        #ax2.set_ylabel("Number of properties")
        """


    
    plt.title(title)
    plt.xticks(rotation=45)
    plt.ylabel("Price (€/m²)")
    plt.show()

In [ ]:
def histogram_top_5_cities(df, title):
   
    top_cities = df["nearest_city"].value_counts().head(5).index
    df_top = df[df["nearest_city"].isin(top_cities)]

    ax= sns.countplot(data=df_top, x="nearest_city", hue="category")

    
    for p in ax.patches:
            height = p.get_height()
        
            if height > 0:
                ax.text(
                    p.get_x() + p.get_width() / 2,
                    height + 10,              
                    int(height),              
                    ha="center",
                    fontsize=9)


    plt.title(title)
    plt.xticks(rotation=45)
    plt.ylabel("Number of properties")
    plt.show()

In [ ]:
def stacked_bar_top_cities(df, title):
    
    top_cities = df["nearest_city"].value_counts().head(5).index

    counts = df.groupby(["nearest_city", "category"]).size().unstack()
    counts = counts.loc[top_cities]
    colors = [epc_colors[2], epc_colors[6]] 

    
    counts.plot(kind="bar", stacked=True, figsize=(10,6))
    ax = counts.plot(kind="bar", stacked=True, figsize=(10,6), color = colors)
    for p in ax.patches:

        height = p.get_height()
        bottom = p.get_y()

        if height > 0:
            ax.text(
                p.get_x() + p.get_width() / 2,
                bottom + height / 2,     
                int(height),
                ha="center",
                va="center",
                fontsize=9,
                color="white"            
        )


    plt.title(title)
    plt.ylabel("Number of properties")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
def scatter_surface_price(df, title):
    plt.figure(figsize = (10,6))

    sns.scatterplot(
        data = df,
        x = "livable_surface",
        y = "price",
        hue = "category",
        alpha = 0.5 
    )
    plt.yscale("log")
    ax = plt.gca()
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'€{x/1e6:.1f}M')
    )

    plt.title(title)
    plt.xlabel("Livable Surface (m²)")
    plt.ylabel("Price (€ millions)")
    plt.show()

scatter_surface_price(df_apartment, "Scatter_apartment")

In [ ]:
def plot_interactive_heatmap(df, title):

    corr = df.corr(numeric_only=True)

    corr = corr.dropna(axis=0, how="all").dropna(axis=1, how="all")

    fig = px.imshow(
        corr,
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        title=title
    )

    fig.update_layout(height=700, width=800)

    fig.show()